In [1]:
!pip -q install langchain
!pip -q install langchain-community
!pip -q install sentence-transformers
!pip -q install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 76.2 MB/s eta 0:00:00


In [7]:

from langchain_text_splitters import RecursiveCharacterTextSplitter


from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings


/tmp/ipykernel_2493/1841743165.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [8]:
text = """
Machine Learning is a branch of Artificial Intelligence.

Deep Learning is a subset of Machine Learning.

Regression predicts continuous values.

Classification predicts labels.

Neural Networks are inspired by the human brain.
"""

In [9]:
doc=Document(page_content=text)

In [10]:
splitter=RecursiveCharacterTextSplitter(
    chunk_size=80,
    chunk_overlap=20
)
chunks=splitter.split_documents([doc])

In [11]:
embedding=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_2493/3241142964.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding=HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [12]:
db=FAISS.from_documents(
    chunks,
    embedding
)

In [13]:
retriever=db.as_retriever()

In [15]:
print(type(retriever))

<class 'langchain_core.vectorstores.base.VectorStoreRetriever'>


In [16]:
query="what is machine Learning"
results=retriever.invoke(query)

In [18]:
for doc in results:

    print(doc.page_content)

    print("="*50)

Machine Learning is a branch of Artificial Intelligence.
Deep Learning is a subset of Machine Learning.
Neural Networks are inspired by the human brain.
Regression predicts continuous values.

Classification predicts labels.


In [20]:
results=retriever.invoke(
    "What is Deep Learning?"
)
[res.page_content for res in results]

['Deep Learning is a subset of Machine Learning.',
 'Machine Learning is a branch of Artificial Intelligence.',
 'Neural Networks are inspired by the human brain.',
 'Regression predicts continuous values.\n\nClassification predicts labels.']

# Search Types
there are 3 differents search type in Langchain
1. Similarity(Default)
2. mmr
3. similarity_score_threshold

# Similarity

In [22]:
retriever=db.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k":2
    }
)

results=retriever.invoke(
    "Machine Learning"
)
for res in results:
  print(res.page_content)

Machine Learning is a branch of Artificial Intelligence.
Deep Learning is a subset of Machine Learning.


#MMR

In [29]:

retriever=db.as_retriever(
    search_type='mmr',
    search_kwargs={
        "k":2,
        "fetch_k":5
    }

)
results=retriever.invoke(
    "Machine Learning"
)
for res in results:
  print(res.page_content)

Machine Learning is a branch of Artificial Intelligence.
Regression predicts continuous values.

Classification predicts labels.


fetch_k=5, takes top 5 chunks and mmr selects best-2 chunks(k=2) from top 5

# Thresold

In [28]:
retriever = db.as_retriever(

    search_type="similarity_score_threshold",

    search_kwargs={

        "score_threshold":0.7
    }
)
results=retriever.invoke(
    "Quantum Computing"
)
for res in results:
  print(res.page_content)

/usr/local/lib/python3.12/dist-packages/langchain_core/vectorstores/base.py:1048: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='93906ccf-59c7-41d5-abf3-e31225a4fd82', metadata={}, page_content='Machine Learning is a branch of Artificial Intelligence.'), np.float32(-0.0529114)), (Document(id='6ec461ad-38b8-4cf8-9094-a249b971c798', metadata={}, page_content='Deep Learning is a subset of Machine Learning.'), np.float32(-0.09436691)), (Document(id='f0a6bf5b-498f-497d-8a0d-72b810d9ac5f', metadata={}, page_content='Neural Networks are inspired by the human brain.'), np.float32(-0.11203766)), (Document(id='97c0b90e-7ab6-4e89-99f8-b6ac7daca523', metadata={}, page_content='Regression predicts continuous values.\n\nClassification predicts labels.'), np.float32(-0.35035264))]
  self.vectorstore.similarity_search_with_relevance_scores(


In [30]:
retriever=db.as_retriever(
    search_type='mmr',
    search_kwargs={
        "k":2,
        "fetch_k":5
    }

)
results=retriever.invoke(
    "Quantum Computing"
)
for res in results:
  print(res.page_content)

Machine Learning is a branch of Artificial Intelligence.
Neural Networks are inspired by the human brain.


**For Sensitive applications use similarity_score_thresold**



```
                     Query
                       │
        ┌──────────────┼──────────────┐
        │              │              │
        ▼              ▼              ▼
 similarity_search  similarity_    retriever
                    search_with_     .invoke()
                      score
        │              │              │
        ▼              ▼              ▼
   Documents      (Document,Score)  Documents
```

